# Aprendendo MDAnalysis

Notebook de apoio — não faz parte do exercício, serve para entender a ferramenta antes de usá-la.

**O que o MDAnalysis é:** uma biblioteca para ler trajetórias de dinâmica molecular e fazer perguntas sobre elas. Ela abstrai as dezenas de formatos de arquivo que existem (`.xyz`, `.dcd`, `.xtc`, `.pdb`, `.trr`…) atrás de uma interface só.

**O conceito central** é o `Universe`, que junta duas coisas de naturezas diferentes:

| | o que é | muda com o tempo? |
|---|---|---|
| **topologia** | quem são os átomos, seus nomes, massas, quem está ligado a quem | não |
| **trajetória** | onde cada átomo está, frame a frame | sim |

Quase toda confusão com MDAnalysis vem de misturar as duas.

In [ ]:
import numpy as np
import MDAnalysis as mda

print("MDAnalysis", mda.__version__)

---
## 1. Carregar

O caso normal é passar dois arquivos: um de topologia e um de trajetória.

```python
u = mda.Universe("sistema.pdb", "trajetoria.dcd")
```

Aqui só temos um `.xyz`, que é um formato pobre — tem os nomes dos átomos e as coordenadas, e mais nada. Nem ligações, nem resíduos, nem cargas.

Por isso o exercício usa `to_guess`: manda o MDAnalysis **inferir** o que falta a partir das distâncias entre átomos e de raios de van der Waals tabelados.

- `dt=10` informa que cada frame está separado por 10 ps. Sem isso o `time` viria em unidades erradas.

In [ ]:
u = mda.Universe(
    "trajectory.xyz",
    to_guess=["bonds", "angles", "dihedrals", "masses"],
    dt=10,
)

print(u)
print("átomos :", len(u.atoms))
print("frames :", len(u.trajectory))
print("dt     :", u.trajectory.dt, "ps")

### O que o `.xyz` NÃO tem

Vale ver o erro acontecer, para reconhecê-lo depois. Descomente uma linha por vez:

In [ ]:
# u.atoms.types        # NoDataError: não há informação de tipo
# u.residues.resnames  # NoDataError: não há informação de resíduo

print("resíduos:", len(u.residues))

O MDAnalysis enxerga **um resíduo só** — a molécula inteira. Ele não sabe que isso é um hexapeptídeo com seis alaninas.

Consequência prática: seleções tipo `"resid 3"` ou `"protein"` não funcionam aqui. Você vai precisar selecionar por índice de átomo.

---
## 2. Átomos: `AtomGroup`

`u.atoms` é um `AtomGroup` — a estrutura de dados que você mais vai usar. Os atributos são **vetorizados**: pedir `.names` devolve um array com todos de uma vez, não um por um.

In [ ]:
print("nomes  :", u.atoms.names[:10])
print("massas :", np.round(u.atoms.masses[:10], 3))
print("índices:", u.atoms.indices[:10])
print()
print("posições, shape:", u.atoms.positions.shape)   # (n_atoms, 3), em ångström
print(u.atoms.positions[:3])

As massas foram **inferidas** a partir dos nomes (`C` → 12.011). Confira se bate com o que você espera — num modelo de átomo unido, um sítio `CH3` pesa 15, não 12, e o MDAnalysis não tem como saber disso a partir de um nome `C`.

Isso importa para o raio de giro, que é ponderado pela massa.

---
## 3. Seleções

`select_atoms` usa uma mini-linguagem, parecida com a do VMD.

In [ ]:
print("name C      :", len(u.select_atoms("name C")))
print("name N      :", len(u.select_atoms("name N")))
print("name C or name N:", len(u.select_atoms("name C or name N")))
print("index 0:5   :", len(u.select_atoms("index 0:5")))   # inclusivo nas duas pontas
print("all         :", len(u.select_atoms("all")))

# também dá para fatiar como lista
print("u.atoms[3:8]:", u.atoms[3:8].names)

---
## 4. A trajetória é um *stream* — este é o ponto que mais confunde

O MDAnalysis **não carrega a trajetória inteira na memória**. Ele mantém um único frame por vez, e `u.atoms.positions` sempre reflete o frame atual.

Trocar de frame é mover um ponteiro.

In [ ]:
u.trajectory[0]
print("frame", u.trajectory.ts.frame, "| tempo", u.trajectory.ts.time, "ps")
print("  átomo 0 em", np.round(u.atoms.positions[0], 3))

u.trajectory[500]
print("frame", u.trajectory.ts.frame, "| tempo", u.trajectory.ts.time, "ps")
print("  átomo 0 em", np.round(u.atoms.positions[0], 3))

O mesmo `u.atoms` devolveu coordenadas diferentes. Ele não é uma foto — é uma janela para o frame atual.

**A consequência prática:** se você quer guardar alguma coisa de vários frames, precisa acumular dentro do laço. Sair do laço e tentar ler de novo só te dá o último frame.

In [ ]:
# jeito certo: acumular durante a iteração
tempos = []
for ts in u.trajectory[:5]:
    tempos.append(ts.time)
    print(f"frame {ts.frame:4d}   t = {ts.time:7.1f} ps   "
          f"átomo 0 = {np.round(u.atoms.positions[0], 2)}")

print("\ntempos coletados:", tempos)

In [ ]:
# Se precisar guardar coordenadas, copie — e prefira acumular em lista, não
# concatenar array a cada passo.
coords = []
for ts in u.trajectory[:5]:
    coords.append(u.atoms.positions.copy())
coords = np.array(coords)
print("shape:", coords.shape)   # (n_frames, n_atoms, 3)

Depois de iterar, o ponteiro fica onde parou. Se a próxima célula depende do frame 0, volte explicitamente com `u.trajectory[0]`. Bug silencioso clássico: calcular uma referência "do primeiro frame" que na verdade é do último.

---
## 5. Conectividade inferida

Como o `.xyz` não traz ligações, o `to_guess` as inventou a partir das distâncias. Ângulos e diedros saem em cascata da lista de ligações.

In [ ]:
print("ligações:", len(u.bonds))
print("ângulos :", len(u.angles))
print("diedros :", len(u.dihedrals))

São muito mais diedros do que os 12 do esqueleto que o exercício quer — a inferência devolve **todos** os caminhos de quatro átomos ligados em sequência, incluindo os que passam por hidrogênios e metilas.

Então você precisa descobrir **quais** dessa lista são os φ e ψ. Para isso, inspecione os átomos de cada um:

In [ ]:
d = u.dihedrals[3]

print("índices dos 4 átomos:", [a.index for a in d.atoms])
print("nomes               :", [a.name for a in d.atoms])
print("valor               :", round(d.value(), 2), "graus")

In [ ]:
# Varrer os primeiros e ver o padrão. Um φ tem a forma C–N–C–C;
# um ψ tem a forma N–C–C–N.
for i in range(12):
    d = u.dihedrals[i]
    idx = [a.index for a in d.atoms]
    nomes = "-".join(a.name for a in d.atoms)
    print(f"{i:3d}  {str(idx):22s} {nomes:12s} {d.value():8.2f}")

In [ ]:
# Todos os valores de uma vez, para o frame atual — bem mais rápido que
# chamar .value() num laço.
vals = u.dihedrals.values()
print(vals.shape, vals[:6].round(2))

> **Cuidado com o `to_guess`:** a lista de diedros depende de raios de van der Waals tabelados e pode mudar entre versões da biblioteca. Se um dia os índices "certos" pararem de bater, é isso. Anote no notebook qual versão você usou.

---
## 6. Métodos prontos

O `AtomGroup` já traz várias quantidades geométricas. Todas se referem ao **frame atual**.

In [ ]:
u.trajectory[0]
g = u.select_atoms("all")

print("massa total       :", round(g.total_mass(), 2))
print("centro de massa   :", np.round(g.center_of_mass(), 3))
print("centro geométrico :", np.round(g.center_of_geometry(), 3))
print("raio de giro      :", round(g.radius_of_gyration(), 4), "Å")

---
## 7. O submódulo `analysis`

Análises que varrem a trajetória inteira ficam em `MDAnalysis.analysis`. Todas seguem o mesmo padrão: construir o objeto, chamar `.run()`, ler `.results`.

In [ ]:
from MDAnalysis.analysis import rms

R = rms.RMSD(u, u, select="all", ref_frame=0)
R.run(stop=5)          # stop=5 só para ver o formato rápido; tire depois

print("colunas: frame, tempo, RMSD")
print(R.results.rmsd.round(4))

O `rms.RMSD` **superpõe** as estruturas antes de comparar, removendo translação e rotação de corpo rígido. Sem isso o RMSD mediria a molécula andando pela caixa em vez da mudança de forma.

Confira sempre: o RMSD do frame de referência contra ele mesmo tem que dar exatamente 0.

Outros módulos úteis do `analysis`:

| módulo | para quê |
|---|---|
| `rms.RMSD`, `rms.RMSF` | desvio estrutural e flutuação por átomo |
| `align.AlignTraj` | alinhar uma trajetória inteira a uma referência |
| `dihedrals.Ramachandran` | φ/ψ automáticos — **exige** informação de resíduo, que o `.xyz` não tem |
| `distances` | matrizes de distância, contatos |

---
## 8. Escrever arquivos

Útil para exportar um subconjunto de frames — por exemplo, só os que pertencem a um cluster.

In [ ]:
selecao = [0, 10, 20, 30]
u.select_atoms("all").write("exemplo_frames.xyz", frames=u.trajectory[selecao])

conferindo = mda.Universe("exemplo_frames.xyz")
print("frames escritos:", len(conferindo.trajectory))

---
## Resumo das armadilhas

1. **`u.atoms.positions` é o frame atual**, não a trajetória. Acumule dentro do laço.
2. **O ponteiro fica onde parou.** Volte com `u.trajectory[0]` antes de qualquer coisa que dependa do primeiro frame.
3. **`.xyz` não tem resíduos nem tipos.** Seleções por `resid`/`protein` não funcionam; use índice.
4. **Massas são inferidas do nome.** Em modelo de átomo unido isso subestima — um sítio `CH3` lido como `C` pesa 12 em vez de 15.
5. **`to_guess` inventa a conectividade** a partir de distâncias, e o resultado depende da versão da biblioteca.
6. **RMSD sem superposição** mede movimento de corpo rígido, não conformação.
7. **Unidades:** ångström e picossegundo, sempre.

## Onde procurar

- Guia do usuário: <https://userguide.mdanalysis.org/stable/>
- Linguagem de seleção: <https://userguide.mdanalysis.org/stable/selections.html>
- Referência da API: <https://docs.mdanalysis.org/stable/>